# Entraînement de l'agent Rocket League sur Kaggle (GPU ou TPU)

**Ce qui tourne où.** RocketSim (la simulation des parties) n'existe que sur CPU. L'accélérateur fait l'**apprentissage**
(GPU T4, ou TPU via PyTorch/XLA) et, en session GPU, éventuellement l'inférence pendant les parties (2e T4).
Au début de chaque session, quelques minutes de **mesures automatiques** choisissent la configuration la plus rapide
sur la machine obtenue (`<run>/autotune.yaml`). Détails : `cloud/README.md`.

**Avant la 1re session** :
1. Sur le portable : `python scripts/make_cloud_bundle.py --init runs/<run de départ> --anchors-from <profils>` (s7f : `--init runs/s7e_spo_laptop --anchors-from s7 s7b s7c s7d s7e s7f`) → `dist/rl-agent-code.zip` et `dist/rl-agent-init.zip`.
2. Créer 3 datasets **privés** : `rl-agent-code` (le zip du code), `rl-collision-meshes` (ton dossier `collision_meshes`), `rl-agent-init` (le zip du point de départ, optionnel).
3. Ce notebook : *Add Input* → les datasets ; *Settings* → **Accelerator : GPU T4 x2** ou **TPU** (pas P100), **Internet : On**.
4. **Save Version → Save & Run All (Commit)** : le notebook tourne seul, navigateur fermé. Les checkpoints sont dans l'onglet *Output*.

**Sessions suivantes** : *Add Input* → *Notebook Output Files* → ce notebook (dernière version) : le run reprend où il s'était arrêté.

In [ ]:
import time; KERNEL_T0 = time.time()         # début de session : l'arrêt propre se cale sur la limite Kaggle
PLATFORM = "auto"                            # "auto" (GPU ou TPU détecté), "kaggle" (GPU), "kaggle_tpu"
MODE = "train"                               # "train" = entraînement ; "bench" = mesures seules (~5 min) puis arrêt
RUN_NAME = "s7f_spo_kaggle"
PROFILES = "s7 s7b s7c s7d s7e s7f"          # réglages d'apprentissage (configs/*.yaml), dans l'ordre
HOURS = "auto"                               # "auto" = arrêt propre 20 min avant la limite (12 h GPU, 9 h TPU), ou nb d'heures
AUTOTUNE = True                              # mesures en début de session (~3-6 min) → configuration la plus rapide
# "auto" = cherché n'importe où sous /kaggle/input (les chemins exacts varient selon les comptes Kaggle).
# On peut aussi mettre un chemin, ex. "/kaggle/input/datasets/<ton-pseudo>/rl-agent-code".
CODE_INPUT = "auto"                          # dossier contenant pyproject.toml
MESHES_INPUT = "auto"                        # dossier contenant soccar/*.cmf
INIT_INPUT = "auto"                          # ckpt.pt + pool/ (utilisé seulement s'il n'y a pas encore de checkpoint)
PREV_OUTPUT = "auto"                         # Output d'une version précédente ajouté en Input → reprise du run
RUNS_DIR = "/kaggle/working/runs"            # sauvegardé dans l'Output de la version

In [ ]:
import glob, os, shutil, subprocess, zipfile

def find_dir_with(root, marker):
    """Dossier sous root qui contient `marker` (gère les zips décompressés ou non)."""
    for z in glob.glob(os.path.join(root, "**", "*.zip"), recursive=True):
        dest = os.path.join("/tmp/unzipped", os.path.basename(z)[:-4])
        if not os.path.exists(dest):
            zipfile.ZipFile(z).extractall(dest)
    for base in (root, "/tmp/unzipped"):
        hits = sorted(glob.glob(os.path.join(base, "**", marker), recursive=True), key=len)
        if hits:
            return os.path.dirname(hits[0])
    return None


def where(setting, marker, root="/kaggle/input"):
    """setting = "auto" (cherche partout sous /kaggle/input) ou un chemin explicite."""
    return find_dir_with(root if setting == "auto" else setting, marker)

code_dir = where(CODE_INPUT, "pyproject.toml")
meshes_dir = where(MESHES_INPUT, "soccar")
init_dir = where(INIT_INPUT, "ckpt.pt") or where(INIT_INPUT, "init.json")  # init.json : ancres seules (run de zéro)
assert code_dir, "code introuvable : ajouter le dataset rl-agent-code en Input (ou mettre son chemin dans CODE_INPUT)"
assert meshes_dir, "collision meshes introuvables : ajouter le dataset rl-collision-meshes (dossier avec soccar/)"
shutil.copytree(code_dir, "/tmp/rl-agent", dirs_exist_ok=True)       # le code ne va pas dans l'Output
os.makedirs(RUNS_DIR, exist_ok=True)
prev_root = "/kaggle/input" if PREV_OUTPUT == "auto" else PREV_OUTPUT
if PREV_OUTPUT and os.path.isdir(prev_root):                          # reprise : on recopie le run précédent
    marks = sorted(glob.glob(os.path.join(prev_root, "**", "runs", RUN_NAME, "latest.txt"), recursive=True))
    target = os.path.join(RUNS_DIR, RUN_NAME)
    if marks and not os.path.isdir(target):
        shutil.copytree(os.path.dirname(marks[-1]), target)
        print("reprise du run précédent :", os.path.dirname(marks[-1]))
    if not marks and not os.path.isdir(target) and not init_dir:     # reprise demandée mais rien trouvé :
        raise SystemExit("reprise demandée, mais aucun run " + RUN_NAME + " dans " + prev_root +
                         " : arrêt plutôt qu'un entraînement de zéro")
print("code   :", code_dir)
print("meshes :", meshes_dir)
print("init   :", init_dir)
!ls /kaggle/input
!nproc; free -g | head -2; nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "pas de GPU NVIDIA" 

In [ ]:
import os
import subprocess
env = dict(os.environ, CODE_DIR="/tmp/rl-agent", MESHES_DIR=meshes_dir, RUNS_DIR=RUNS_DIR, PLATFORM=PLATFORM,
           MODE=MODE, RUN_NAME=RUN_NAME, PROFILES=PROFILES, HOURS=str(HOURS), AUTOTUNE="1" if AUTOTUNE else "0",
           INIT_DIR=init_dir or "", VENV_DIR="/tmp/rlenv", KERNEL_T0=str(int(KERNEL_T0)))
script = "/tmp/rl-agent/cloud/run_cloud.sh"
data = open(script, "rb").read()                     # lire AVANT d'ouvrir en écriture
open(script, "wb").write(data.replace(bytes([13, 10]), bytes([10])))   # fins de ligne Linux
p = subprocess.Popen(["bash", script], env=env, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True)
for line in p.stdout:                      # au premier plan : en mode Commit, la version dure jusqu'à la fin
    if not line.startswith("it ") or line.split("|")[0].strip().endswith("0"):
        print(line, end="", flush=True)    # toutes les 10 itérations + messages
p.wait()
print("code de sortie :", p.returncode)

In [ ]:
import json, os
tune = f"{RUNS_DIR}/{RUN_NAME}/autotune.yaml"
if os.path.exists(tune):
    print(open(tune).read())
path = f"{RUNS_DIR}/{RUN_NAME}/metrics.jsonl"
if MODE == "train" and os.path.exists(path):
    last = [json.loads(l) for l in open(path)][-1]
    print(f"steps {last['global_step']:,.0f} | {last['sps_total']:,.0f} steps/s (collecte {last['sps_collect']:,.0f}, "
          f"update {last['time/learn_s']:.1f} s) | elo {last['elo']:.0f} | entropie {last['learn/entropy']:.3f}")
    for k in sorted(k for k in last if k.startswith(("drill/", "mech/", "vs_anchor/", "curriculum/")) and not k.endswith("episodes")):
        print(f"{k:35s} {last[k]:.3f}")
!ls -la {RUNS_DIR}/{RUN_NAME}/checkpoints 2>/dev/null || true